# Build index to s3 objects
s3 objects are equivalent to files.  They are just stored and managed in a different way than file systems.   This notebook demonstrates how to build an index of the holdings of data by Earthscope for the test TA data.   

Earthscope organizes their waveform in miniseed day volumes.   The day volumes are held in the s3 equivalent of a directory which they call a "prefix".   This notebook creates a MongoDB collection with the name "wf_s3" 
that will be used to drive processing to extract event data.   This notebook is indepent of that and more-or-less builds a directory of all file-like (s3 objects) linked to TA data for all of the year 2010.   Our test data does not cover the entire year, but we do the entire year to demonstrate that creating the index is very lightweight.  Fetching waveforms using it is not and is done in a different notebook,

In [1]:
from mspasspy.client import Client
mspass_client=Client()
dask_client = mspass_client.get_scheduler()
db = mspass_client.get_database("ES2026")
dask_client

/opt/conda/lib/python3.10/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36867 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/auth0%7C66620230da0803120dd94aaf/proxy/36867/status,
Dashboard: /user/auth0%7C66620230da0803120dd94aaf/proxy/36867/status,Workers: 4
Total threads: 4,Total memory: 29.08 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40487,Workers: 0
Dashboard: /user/auth0%7C66620230da0803120dd94aaf/proxy/36867/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:46527,Total threads: 1
Dashboard: /user/auth0%7C66620230da0803120dd94aaf/proxy/42215/status,Memory: 7.27 GiB
Nanny: tcp://127.0.0.1:37215,


2026-06-17 11:48:53,711 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:45119'.
2026-06-17 11:48:53,714 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:45213'.
2026-06-17 11:48:53,715 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:41541'.


In [7]:
import boto3
from botocore.config import Config
from earthscope_sdk import EarthScopeClient

from s3_worker_plugin import fetch_s3_client

client = EarthScopeClient()
creds = client.user.get_aws_credentials()

S3_ACCESS_POINT = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
BUCKET = S3_ACCESS_POINT

session = boto3.Session(
    aws_access_key_id=creds.aws_access_key_id,
    aws_secret_access_key=creds.aws_secret_access_key.get_secret_value(),
    aws_session_token=creds.aws_session_token.get_secret_value(),
)

s3_client = fetch_s3_client(session)


In [8]:
def fetch_net_day_list(s3_client,net,year,day,server_data_key="server_data")->tuple:
    """
    Fetch the raw list of object names from s3 for network net for year and julian day day.  
    Returns a tuple/array of length 2.  Comp 0 is a list of s3 object names for that 
    net, year, day combination.  Comp 1 is a dictionary returned by aws of the attribute
    they call "HPPTHeader".   The main use of that is to define the server and aws region
    from which the data can be fetched.   That is contant for this tutorial using Earthscope 
    data only, but it is useful to include it as access from multiple servers would need that data.
    """
    base_prefix = "miniseed"
    prefix = f"{base_prefix}/{net}/{year}/{day:03d}/"
    # works for now - BUCKET needs to be an arg
    list_resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=prefix, Delimiter="/")
    keys=[]
    for obj in list_resp['Contents']:
        rawkey = obj['Key']
        k = rawkey.split("#")
        keys.append(k[0])
    server_data = list_resp['ResponseMetadata']['HTTPHeaders']
    # load as a subdoc with this key
    auxmd = {server_data_key : server_data}
    return [keys,auxmd]

In [9]:
from obspy import UTCDateTime
def objlist2doclist(objlist,auxmd=None):
    """
    Splits s3 object names into dictionary entries to create a document to be saved in MongoDB. 
    Use dictionary like object *auxmd* to load additional, constant metadata to each doc returned.
    """
    doclist=list()
    for objname in objlist:
        doc=dict()
        doc["s3key"] = objname
        s = objname.split("/")
        doc["net"] = s[1]
        yr = int(s[2])
        doc["year"] = yr
        jday = int(s[3])
        doc["jday"] = jday
        starttime = UTCDateTime(year=yr,julday=jday)
        doc["starttime"] = starttime.timestamp
        endtime = starttime.timestamp + 86400.0
        doc["endtime"] = endtime
        s2 = s[4].split(".")
        doc["sta"] = s2[0]
        if auxmd is not None:
            for k in auxmd.keys():
                # this maybe should warn if k overwrites an existing key
                doc[k] = auxmd[k]
        doclist.append(doc)
    return doclist
        

In [10]:
db = mspass_client.get_database("ES2026")
netlist=db.arrival_css30.distinct('net')
print(netlist)

['AZ', 'CI', 'IU', 'TA', 'US']


In [11]:
# jdays list - simple for this prototype; just hard code it
import numpy as np
jdays=np.zeros(365,dtype=int)
for i in range(365):
    jdays[i] = i+1

Now use the functions we defined above to create a document for each s3 object with auxmd added.   Save the resuls to MongoDB with the collection name "wf_s3".   

In [12]:
import time 
t0=time.time()
prefix_list=list()
for net in netlist:
    for day in jdays:
        s3names,auxmd = fetch_net_day_list(s3_client,net,2010,day)
        doclist=objlist2doclist(s3names,auxmd)
        db.wf_s3.insert_many(doclist)
t=time.time()
n=db.wf_s3.count_documents({})
print("Total number of documents now in wf_s3=",n)
print("Elapsed time to build index=",t-t0)

Total number of documents now in wf_s3= 225486
Elapsed time to build index= 89.54759955406189


Note that process is fast enough there is no need to add the complexity of running it in parallel.   It looks like one could build an index for the entire TA data set in less than 30 minutes running serial.  

Before leaving, verify what a typical document looks like with the mspass pretty print function.

In [13]:
from mspasspy.util.seismic import print_metadata
doc=db.wf_s3.find_one()
print_metadata(doc)

{
  "_id": {
    "$oid": "6a327a659fb54d85e6e2faf6"
  },
  "s3key": "miniseed/AZ/2010/001/BZN.AZ.2010.001",
  "net": "AZ",
  "year": 2010,
  "jday": 1,
  "starttime": 1262304000.0,
  "endtime": 1262390400.0,
  "sta": "BZN",
  "server_data": {
    "x-amzn-requestid": "9067c7d3-78ce-4483-bd5d-afedfdb77d62",
    "server": "AmazonS3",
    "x-amz-bucket-region": "us-east-2",
    "x-amz-request-id": "BKT30ABX9DD4CW8A",
    "content-type": "application/xml",
    "x-amz-id-2": "UoKUpaCwNaDnrLhsSdz5oErcCrN6B46oCZXQdw3YeezAFBhLk1D7mmj4Y0hnrjsTpYiN9Cb5NcglDO5fRNtKbP6LbZ39OjTM",
    "date": "Wed, 17 Jun 2026 10:43:49 GMT",
    "transfer-encoding": "chunked",
    "connection": "keep-alive"
  }
}


For this algorithm this index will speed processing enormously.

In [14]:
from pymongo import ASCENDING
db.wf_s3.create_index([
       ("year",ASCENDING),
       ("jday",ASCENDING),
])

'year_1_jday_1'

2026-06-17 11:48:53,707 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/distributed/comm/tcp.py", line 226, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/distributed/worker.py", line 1273, in heartbeat
    response = await retry_operation(
  File "/opt/conda/lib/python3.10/site-packages/distributed/utils_comm.py", line 416, in retry_operation
    return await retry(
  File "/opt/conda/lib/python3.10/site-packages/distributed/utils_comm.py", line 395, in retry
    return await coro()
  File "/opt/conda/lib/python3.10/site-packages/distributed/core.py", line 1259, in send_recv_from_rpc
    return await send_recv(comm=comm, op=key, **kwarg